In [4]:
# 1. Clone your GitHub repository into Colab
!git clone https://github.com/daljeetkaurJohar/qm640-governance-analytics.git

# 2. Change directory into your cloned project folder
%cd qm640-governance-analytics

# 3. List all files to make sure everything downloaded
!ls -la

Cloning into 'qm640-governance-analytics'...
remote: Enumerating objects: 394, done.
remote: Counting objects: 100% (184/184), done.
remote: Compressing objects: 100% (169/169), done.
remote: Total 394 (delta 94), reused 35 (delta 14), pack-reused 210 (from 2)
Receiving objects: 100% (394/394), 11.23 MiB | 15.57 MiB/s, done.
Resolving deltas: 100% (160/160), done.
/content/qm640-governance-analytics
total 48
drwxr-xr-x 8 root root 4096 Sep  1 04:25 .
drwxr-xr-x 1 root root 4096 Sep  1 04:25 ..
drwxr-xr-x 4 root root 4096 Sep  1 04:25 data
drwxr-xr-x 2 root root 4096 Sep  1 04:25 docs
drwxr-xr-x 2 root root 4096 Sep  1 04:25 figures
drwxr-xr-x 8 root root 4096 Sep  1 04:25 .git
drwxr-xr-x 2 root root 4096 Sep  1 04:25 notebooks
-rw-r--r-- 1 root root 8959 Sep  1 04:25 README.md
drwxr-xr-x 2 root root 4096 Sep  1 04:25 reports
-rw-r--r-- 1 root root 1145 Sep  1 04:25 requirements.txt


In [5]:
import os

print("==========================================")
print(" Folder Directory Check ")
print("==========================================\n")

for root, dirs, files in os.walk("."):
    # Ignore hidden git folders
    if ".git" in root:
        continue

    level = root.replace(".", "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}📁 {os.path.basename(root)}/")

    sub_indent = " " * 4 * (level + 1)
    for f in files:
        if not f.startswith("."):  # skip hidden files
            print(f"{sub_indent}📄 {f}")

print("\n==========================================")

 Folder Directory Check 

📁 ./
    📄 README.md
    📄 requirements.txt
    📁 notebooks/
        📄 RQ1_6_Interpretation_and_Robustness.ipynb
        📄 04_nasa_promise_data_loading.ipynb
        📄 RQ4_2_EDA.ipynb
        📄 RQ1_6_Interpretation_and_Robustness_UPDATED.ipynb
        📄 RQ2_External_validation.ipynb
        📄 RQ1_2_Hadoop.ipynb
        📄 05_nasa_promise_cleaning_eda_baseline_model.ipynb
        📄 RQ3_1_PCAOB_Extraction.ipynb
        📄 RQ1_3_Kafka.ipynb
        📄 RQ3_2_Model_Training_v2.ipynb
        📄 RQ5_2_Bootstrap_CI.ipynb
        📄 RQ2_3_Analysis.ipynb
        📄 RQ2__full_reassignments.ipynb
        📄 RQ2_4_EDA.ipynb
        📄 RQ2_1_JIRA_Extraction.ipynb
        📄 RQ3_4_EDA.ipynb
        📄 RQ3_3_SHAP.ipynb
        📄 RQ2_2_Reassignments.ipynb
        📄 06_nasa_promise_statistical_testing.ipynb
        📄 RQ1_4_Tika.ipynb
        📄 RQ5_1_Point_Estimate_Synthesis.ipynb
        📄 RQ5_3_EDA.ipynb
        📄 RQ1_5_EDA_Complete_Analysis.ipynb
        📄 RQ1_1_Camel.ipynb
        📄 R

In [6]:
import os
import glob
import pandas as pd

def scan_data_folder():
    print("==========================================")
    print(" Starting Data Integrity & Verification ")
    print("==========================================\n")

    # Search for all CSV files under data/ or the current directory
    csv_files = glob.glob("data/**/*.csv", recursive=True) + glob.glob("*.csv")

    if not csv_files:
        print("⚠️ No CSV files found! Make sure your data is placed inside the 'data/' folder.")
        return

    print(f"Found {len(csv_files)} CSV file(s):\n")

    for file_path in sorted(csv_files):
        try:
            df = pd.read_csv(file_path)
            print(f"✓ {file_path}")
            print(f"   ↳ Total Rows: {len(df):,} | Total Columns: {len(df.columns)}")

            # Check for RQ3 Leakage
            if 'audit_area' in [c.lower() for c in df.columns]:
                print("   ⚠️  ALERT: 'audit_area' column found. Drop this feature in RQ3 pipelines to prevent leakage.")

            # Check for RQ4 Event Column
            if 'event_observed' in df.columns:
                remediated = df['event_observed'].sum()
                total = len(df)
                print(f"   ✓ Censoring Breakdown: {remediated}/{total} remediated ({(remediated/total)*100:.1f}%)")

            print("-" * 45)
        except Exception as e:
            print(f"✗ Error reading {file_path}: {e}")

    print("\n==========================================")
    print(" Verification Complete! ")
    print("==========================================")

if __name__ == "__main__":
    scan_data_folder()

 Starting Data Integrity & Verification 

Found 30 CSV file(s):

✓ data/cleaned/apache_jira_raw.csv
   ↳ Total Rows: 30,996 | Total Columns: 7
---------------------------------------------
✓ data/cleaned/audit_disclosure_dataset.csv
   ↳ Total Rows: 16,704 | Total Columns: 12
---------------------------------------------
✓ data/cleaned/camel_real_mined_dataset.csv
   ↳ Total Rows: 12,598 | Total Columns: 9
---------------------------------------------
✓ data/cleaned/hadoop_mining_checkpoint.csv
   ↳ Total Rows: 5,906 | Total Columns: 9
---------------------------------------------
✓ data/cleaned/hadoop_real_mined_dataset.csv
   ↳ Total Rows: 6,038 | Total Columns: 9
---------------------------------------------
✓ data/cleaned/kafka_jira_raw.csv
   ↳ Total Rows: 11,244 | Total Columns: 8
---------------------------------------------
✓ data/cleaned/kafka_mining_progress.csv
   ↳ Total Rows: 433 | Total Columns: 9
---------------------------------------------
✓ data/cleaned/kafka_real_min

In [9]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. Load the two datasets
df_qa = pd.read_csv("data/cleaned/qa_defect_dataset.csv")
df_reassign = pd.read_csv("data/cleaned/num_reassignments_FULL.csv")

print(f"QA Dataset Rows: {len(df_qa):,}")
print(f"Reassignments Dataset Rows: {len(df_reassign):,}")

# 2. Merge on issue ID
# (Check common column for merging, usually 'issue_id' or 'key')
id_col = 'issue_id' if 'issue_id' in df_reassign.columns else df_reassign.columns[0]
df_merged = pd.merge(df_qa, df_reassign, left_on='issue_id', right_on=id_col, how='inner')

# Identify reassignment column name in merged data
reassign_col = [c for c in df_reassign.columns if c != id_col][0] if len(df_reassign.columns) > 1 else 'reassignments'

print(f"✓ Merged successfully! Total matched records: {len(df_merged):,}")
print(f"Reassignment Column: '{reassign_col}' | Duration Column: 'resolution_time_days'\n")

# 3. Clean non-positive durations for Gamma model
df_clean = df_merged[df_merged['resolution_time_days'] > 0].copy()

# 4. Run Gamma GLM model
formula_str = f"resolution_time_days ~ {reassign_col} + C(priority) + C(era)"
print(f"Running GLM with formula: {formula_str}\n")

gamma_model = smf.glm(
    formula=formula_str,
    data=df_clean,
    family=sm.families.Gamma(link=sm.families.links.Log())
).fit()

print("==========================================")
print(" RQ2 Model Refinement: Gamma GLM Results ")
print("==========================================")
print(gamma_model.summary().tables[1])

QA Dataset Rows: 30,733
Reassignments Dataset Rows: 27,385
✓ Merged successfully! Total matched records: 27,137
Reassignment Column: 'num_reassignments' | Duration Column: 'resolution_time_days'

Running GLM with formula: resolution_time_days ~ num_reassignments + C(priority) + C(era)

 RQ2 Model Refinement: Gamma GLM Results 
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                   3.2489      0.136     23.871      0.000       2.982       3.516
C(priority)[T.Critical]     0.6518      0.216      3.011      0.003       0.228       1.076
C(priority)[T.Major]        1.1371      0.120      9.465      0.000       0.902       1.373
C(priority)[T.Minor]        1.1779      0.125      9.437      0.000       0.933       1.423
C(priority)[T.Trivial]      1.1560      0.197      5.882      0.000       0.771       1.541
C(era)[T.pre_ai]           